# Day 074 — Exercise 1: get_video_info

**What you'll build:** `get_video_info(source, info_fn=None) -> dict` — extract video metadata using OpenCV VideoCapture.

**Why it matters:** fps, frame_count, width, and height are needed by every downstream operation: frame extraction, VideoWriter, and duration-based sampling.

In [ ]:
from pathlib import Path

def _make_test_frames(n=10, height=32, width=32):
    import numpy as np
    frames = []
    for i in range(n):
        frame = np.zeros((height, width, 3), dtype=np.uint8)
        frame[:, :, 0] = int(255 * i / max(n - 1, 1))
        frames.append(frame)
    return frames


In [ ]:
_MOCK_META = {
    'fps': 30.0, 'frame_count': 10, 'width': 32, 'height': 32, 'duration_sec': 0.333,
}
_mock_info_fn    = lambda source: dict(_MOCK_META)
_mock_capture_fn = lambda source: _make_test_frames(10)
_mock_writer_fn  = lambda frames, path, fps: (Path(path).write_bytes(b'VIDEO' + bytes(len(frames))), Path(path))[1]
_mock_ffmpeg_fn  = lambda args: {'returncode': 0, 'stdout': '', 'stderr': ''}


## Task

Return a dict with `fps`, `frame_count`, `width`, `height`, `duration_sec`.

- Mock: `if info_fn is not None: return info_fn(source)`
- Real: `import cv2`, `VideoCapture`, `.get(CAP_PROP_*)`, `.release()`
- `duration_sec = round(frame_count / fps, 3) if fps > 0 else 0.0`

## Your Implementation

In [ ]:
def get_video_info(source, info_fn=None) -> dict:
    """Return video metadata: fps, frame_count, width, height, duration_sec.

    Args:
        source:  path to video file (str or Path)
        info_fn: callable(source) -> dict for testing (no OpenCV needed)
    """
    raise NotImplementedError


In [ ]:
def get_video_info(source, info_fn=None):
    if info_fn is not None:
        return info_fn(source)
    import cv2
    cap = cv2.VideoCapture(str(source))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {source}')
    fps         = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = round(frame_count / fps, 3) if fps > 0 else 0.0
    return {
        'fps': fps, 'frame_count': frame_count,
        'width': width, 'height': height, 'duration_sec': duration,
    }


## Automated checks

In [ ]:

score, total = 0, 5
try:
    meta = get_video_info('video.mp4', info_fn=_mock_info_fn)

    # returns a dict
    assert isinstance(meta, dict)
    score += 1; print("✅ returns a dict")

    # required keys present
    for k in ('fps', 'frame_count', 'width', 'height', 'duration_sec'):
        assert k in meta, f"missing key '{k}'"
    score += 1; print("✅ all required keys present")

    # fps is float
    assert isinstance(meta['fps'], float) and meta['fps'] > 0
    score += 1; print("✅ fps is positive float")

    # frame_count and dimensions are int
    assert isinstance(meta['frame_count'], int)
    assert isinstance(meta['width'], int) and isinstance(meta['height'], int)
    score += 1; print("✅ frame_count, width, height are int")

    # duration_sec is float and approximately correct
    expected = round(meta['frame_count'] / meta['fps'], 3)
    assert abs(meta['duration_sec'] - expected) < 0.01, \
        f"duration_sec {meta['duration_sec']} != expected {expected}"
    score += 1; print("✅ duration_sec = frame_count / fps (correct)")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def get_video_info(source, info_fn=None):
    if info_fn is not None:
        return info_fn(source)
    import cv2
    cap = cv2.VideoCapture(str(source))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {source}')
    fps         = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = round(frame_count / fps, 3) if fps > 0 else 0.0
    return {
        'fps': fps, 'frame_count': frame_count,
        'width': width, 'height': height, 'duration_sec': duration,
    }
```

**Why `int(cap.get(...))`?** The `cap.get()` method always returns a float, even for integer-valued properties. `int()` converts to the expected Python type for frame count and pixel dimensions.

</details>